# Sampling Methods: Inverse-CDF, Rejection, Importance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/sampling_methods_basics.ipynb)

Companion notebook to the [blog post](https://sesen.ai/blog/sampling-methods-inverse-cdf-rejection-importance). Five basic sampling methods from Bishop §11.1 implemented in plain NumPy: inverse-CDF, Box-Muller, rejection sampling, importance sampling, sampling-importance-resampling.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gamma as Gamma_fn

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.spines.top': False, 'axes.spines.right': False})


## 1. Inverse-CDF: exponential samples

If `z ~ Uniform(0, 1)`, then `y = h^{-1}(z)` follows the target distribution, where `h` is its CDF. For the exponential, `h(y) = 1 - exp(-lambda * y)`, giving `y = -log(1 - z) / lambda`.


In [ ]:
lam = 1.0
u = rng.uniform(0, 1, 5000)
y = -np.log1p(-u) / lam

fig, ax = plt.subplots()
ax.hist(y, bins=40, density=True, alpha=0.6, label='inverse-CDF samples')
y_grid = np.linspace(0, 8, 200)
ax.plot(y_grid, lam * np.exp(-lam * y_grid), lw=2, label='target PDF')
ax.set_xlabel('y'); ax.set_ylabel('density'); ax.legend()
ax.set_title(f'Inverse-CDF: Exponential(lambda={lam}), n={len(y)}')
plt.show()


## 2. Box-Muller: standard Gaussian samples

The Gaussian CDF has no closed-form inverse. Box-Muller draws pairs uniformly inside the unit circle, then applies a polar transform that maps each pair to two independent N(0, 1) draws.


In [ ]:
n_target = 4000
z1 = rng.uniform(-1, 1, n_target * 4)
z2 = rng.uniform(-1, 1, n_target * 4)
r2 = z1**2 + z2**2
keep = (r2 > 0) & (r2 <= 1)
z1, z2 = z1[keep][:n_target], z2[keep][:n_target]
r2 = z1**2 + z2**2
factor = np.sqrt(-2 * np.log(r2) / r2)
y1, y2 = z1 * factor, z2 * factor

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(z1[:1500], z2[:1500], s=4, alpha=0.6)
theta = np.linspace(0, 2 * np.pi, 200)
ax1.plot(np.cos(theta), np.sin(theta), 'k-', lw=1.5)
ax1.set_aspect('equal'); ax1.set_xlim(-1.1, 1.1); ax1.set_ylim(-1.1, 1.1)
ax1.set_xlabel('z1'); ax1.set_ylabel('z2'); ax1.set_title('Uniform inside unit circle')

bins = np.linspace(-4, 4, 50)
ax2.hist(y1, bins=bins, density=True, alpha=0.65, label='Box-Muller y1')
x = np.linspace(-4, 4, 200)
ax2.plot(x, stats.norm.pdf(x), lw=2, label='N(0, 1)')
ax2.set_xlabel('y'); ax2.set_ylabel('density'); ax2.set_title('Output: standard normal')
ax2.legend()
plt.tight_layout(); plt.show()
print(f'mean = {y1.mean():.3f}, var = {y1.var():.3f}')


## 3. Rejection sampling: gamma target via scaled Cauchy

Bishop's worked example. Target is `Gamma(a=2, b=1)`. Proposal is a scaled Cauchy centred at `c = a-1` with width `bq^2 = 2a-1`. Pick `k` so that `k * cauchy(z) >= gamma(z)` everywhere.


In [ ]:
a, b = 2.0, 1.0
c, bq = a - 1, np.sqrt(2 * a - 1)

def gamma_pdf(z):
    z = np.asarray(z, dtype=float)
    out = np.zeros_like(z)
    m = z > 0
    out[m] = (b**a) * z[m]**(a-1) * np.exp(-b*z[m]) / Gamma_fn(a)
    return out

def cauchy_unnorm(z):
    return 1.0 / (1 + (z - c)**2 / bq**2)

z_grid = np.linspace(0.01, 30, 5000)
k = (gamma_pdf(z_grid) / cauchy_unnorm(z_grid)).max() * 1.001
print(f'k = {k:.4f}')

u = rng.uniform(0, 1, 2000)
z_cand = c + bq * np.tan(np.pi * (u - 0.5))
z_cand = z_cand[z_cand > 0][:1000]
u_vert = rng.uniform(0, k * cauchy_unnorm(z_cand))
accept = u_vert <= gamma_pdf(z_cand)
samples = z_cand[accept]
print(f'accepted {accept.sum()} / {len(z_cand)} = {accept.mean():.1%}')

fig, ax = plt.subplots(figsize=(8, 4))
z_plot = np.linspace(0, 18, 400)
ax.plot(z_plot, k * cauchy_unnorm(z_plot), lw=2, label='k*q(z) envelope')
ax.plot(z_plot, gamma_pdf(z_plot), lw=2, label='gamma target')
ax.scatter(z_cand[accept], u_vert[accept], s=8, alpha=0.7, c='green', label='accepted')
ax.scatter(z_cand[~accept], u_vert[~accept], s=8, alpha=0.4, c='red', label='rejected')
ax.set_xlim(0, 18); ax.set_xlabel('z'); ax.set_ylabel('density'); ax.legend()
ax.set_title(f'Rejection sampling, accept rate = {accept.mean():.1%}')
plt.show()


### Curse of dimensionality

If the target is `N(0, sigma_p^2 * I)` and the proposal is `N(0, sigma_q^2 * I)`, the optimal `k = (sigma_q/sigma_p)^D` and acceptance rate `~ 1/k`. Even a 5% wider proposal collapses fast in high D.


In [ ]:
sigma_q_over_p = 1.05
Ds = np.arange(1, 101)
accept_rate = (1.0 / sigma_q_over_p) ** Ds

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(Ds, accept_rate, lw=2)
ax.axhline(1e-3, color='gray', lw=1, ls='--')
ax.set_xlabel('dimensionality D'); ax.set_ylabel('acceptance rate (log)')
ax.set_title(f'Rejection sampling acceptance vs dimension (sigma_q / sigma_p = {sigma_q_over_p})')
plt.show()


## 4. Importance sampling: estimate E[z^2] under N(0, 1)

Self-normalised IS. Sample from `q = N(0, sigma_q^2)`, weight by `p(z) / q(z)`, average.


In [ ]:
def importance_estimate(sigma_q, n=5000, target_fn=lambda z: z**2):
    z = rng.normal(0, sigma_q, n)
    log_w = stats.norm.logpdf(z, 0, 1) - stats.norm.logpdf(z, 0, sigma_q)
    w = np.exp(log_w - log_w.max())  # numerical stability
    estimate = np.sum(w * target_fn(z)) / np.sum(w)
    ess = (w.sum() ** 2) / (w ** 2).sum()
    return estimate, ess, len(z)

for sigma_q in [0.5, 1.0, 1.5, 2.0, 3.0]:
    est, ess, n = importance_estimate(sigma_q)
    print(f'sigma_q = {sigma_q}: estimate = {est:.4f} (true = 1), ESS / n = {ess/n:.2f}')


Boxplot across 200 reps: tail-thinner proposals (`sigma_q = 0.5`) give biased and noisy estimates, tail-heavier proposals are stable.


In [ ]:
n, n_reps = 2000, 200
sigmas = [0.5, 1.0, 1.5, 2.0, 3.0]
results = {}
for sq in sigmas:
    ests = []
    for _ in range(n_reps):
        z = rng.normal(0, sq, n)
        log_w = stats.norm.logpdf(z, 0, 1) - stats.norm.logpdf(z, 0, sq)
        w = np.exp(log_w - log_w.max())
        ests.append(np.sum(w * z**2) / np.sum(w))
    results[sq] = np.array(ests)

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([results[s] for s in sigmas], positions=range(len(sigmas)),
           widths=0.55, showfliers=False)
ax.axhline(1.0, color='C1', lw=2)
ax.set_xticks(range(len(sigmas))); ax.set_xticklabels([str(s) for s in sigmas])
ax.set_xlabel('proposal sigma_q (target sigma_p = 1)')
ax.set_ylabel('IS estimate of E[z^2]')
ax.set_title(f'Importance sampling variance (n = {n}, {n_reps} reps)')
plt.show()


## 5. Sampling-importance-resampling (SIR)

Step 1: draw samples from `q`. Step 2: compute normalised weights. Step 3: resample with replacement using those weights. The resampled set approximates `p`.


In [ ]:
sigma_q = 2.0
z = rng.normal(0, sigma_q, 500)
log_w = stats.norm.logpdf(z, 0, 1) - stats.norm.logpdf(z, 0, sigma_q)
w = np.exp(log_w - log_w.max())
w /= w.sum()
idx = rng.choice(np.arange(len(z)), size=len(z), p=w, replace=True)
z_sir = z[idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
z_grid = np.linspace(-6, 6, 300)
ax1.plot(z_grid, stats.norm.pdf(z_grid, 0, 1), lw=2, label='target p')
ax1.plot(z_grid, stats.norm.pdf(z_grid, 0, sigma_q), '--', lw=2, label='proposal q')
ax1.scatter(z, stats.norm.pdf(z, 0, sigma_q), s=3 + 800 * w, alpha=0.5,
            label='samples (size = weight)')
ax1.set_xlim(-6, 6); ax1.set_xlabel('z'); ax1.set_ylabel('density')
ax1.set_title('Step 2: weighted samples'); ax1.legend(fontsize=9)

bins = np.linspace(-6, 6, 40)
ax2.hist(z_sir, bins=bins, density=True, alpha=0.6, label='SIR resampled')
ax2.plot(z_grid, stats.norm.pdf(z_grid, 0, 1), lw=2, label='target p')
ax2.set_xlim(-6, 6); ax2.set_xlabel('z')
ax2.set_title('Step 3: resampled draws approximate p'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

ess = 1.0 / (w**2).sum()
print(f'ESS = {ess:.0f} / {len(z)}, unique resampled = {len(np.unique(z_sir))}')


## Exercises

1. **Inverse-CDF for the Cauchy distribution.** Derive `h^{-1}(z)` for `Cauchy(0, 1)` (PDF `1/(pi*(1+y^2))`) and implement a sampler. Compare your samples to `scipy.stats.cauchy.rvs` via a Q-Q plot.

2. **Tighten the gamma envelope.** Bishop fixes `k` based on a Cauchy. Try a Gaussian proposal with optimised mean and variance instead. Compute the optimal `k` numerically. Does the acceptance rate improve?

3. **Importance sampling for a heavy-tailed target.** Estimate `E[|z|]` under a Cauchy target using a Gaussian proposal. Watch the ESS collapse. Now use a Student-t proposal with 3 degrees of freedom. How much does ESS recover?

4. **SIR with a poor proposal.** Run the SIR demo with `sigma_q = 0.5` (thinner-tailed than the target). Plot a histogram of the resampled draws. Compare to the target. What goes wrong?

5. **Adaptive rejection sketch.** For `p(z) = N(0, 1)`, build a piecewise-linear envelope of `ln p` from tangent lines at three grid points. Plot the envelope. Then exponentiate and sample from the resulting piecewise-exponential by inverse-CDF. Use it as the proposal for plain rejection sampling and report the acceptance rate.


## References

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*, chapter 11.1.
- Gilks, W. R., & Wild, P. (1992). Adaptive rejection sampling for Gibbs sampling. *Applied Statistics*, 41(2), 337-348.
- Andrieu, C., de Freitas, N., Doucet, A., & Jordan, M. I. (2003). An introduction to MCMC for machine learning. *Machine Learning*, 50, 5-43.
- Robert, C. P., & Casella, G. (1999). *Monte Carlo Statistical Methods*. Springer.
